
# Auditoria empírica dos 10.000 vetores \(\theta\)

Este notebook usa **somente o banco já existente** do problema fixo:

```text
results/merged.pkl
```

Ele não usa o notebook do Transformer, não repete o COBYLA e não reconstrói os 10.000 vetores.

As colunas esperadas são:

- `best_parameters`: vetor final com 30 parâmetros;
- `objective_function_value`: energia final;
- `most_frequent_bitstring`: bitstring dominante.

O objetivo é responder, para cada parâmetro:

> **Em quais regiões angulares esse \(\theta_j\) aparece quando a energia é melhor e quando o bitstring ótimo é encontrado?**

Importante: esta é uma **análise de associação** usando as 10.000 execuções já prontas. Como os 30 parâmetros variaram juntos durante o COBYLA, o resultado não prova causalidade isolada. Ele serve para identificar quais parâmetros merecem um teste controlado posterior.


In [ ]:

# ============================================================
# 1. IMPORTAÇÕES E CONFIGURAÇÃO
# ============================================================

from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Arquivo gerado pelo experimento de 10.000 execuções.
DATA_PATH = Path("results/merged.pkl")

# Colunas do banco.
THETA_COLUMN = "best_parameters"
ENERGY_COLUMN = "objective_function_value"
BITSTRING_COLUMN = "most_frequent_bitstring"

# Referências já conhecidas para este problema fixo.
EXACT_ENERGY = -0.8181062577496281
EXACT_BITSTRING = "1001011000"

# Configuração simples da análise.
N_ANGLE_BINS = 24
MIN_RUNS_PER_BIN = 30
NEAR_OPTIMAL_TOL = 1e-3
TOP_THETAS = 6

OUTPUT_DIR = Path("outputs_theta_energy_bitstring_10000")
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (13, 7),
    "figure.dpi": 120,
    "axes.titlesize": 15,
    "axes.labelsize": 12,
    "font.size": 11,
})

print("Arquivo esperado:", DATA_PATH)
print("Saída:", OUTPUT_DIR.resolve())



## 2. Carregar e padronizar o banco

Cada vetor é convertido para uma linha da matriz:

\[
\Theta\in\mathbb{R}^{N\times 30}.
\]

Como os parâmetros são ângulos, todos são envolvidos em:

\[
[-\pi,\pi).
\]


In [ ]:

# ============================================================
# 2. CARREGAMENTO
# ============================================================

def parse_theta(value):
    """Converte uma célula do banco em um vetor NumPy simples."""
    if isinstance(value, np.ndarray):
        return value.astype(float).ravel()

    if isinstance(value, (list, tuple, pd.Series)):
        return np.asarray(value, dtype=float).ravel()

    if isinstance(value, str):
        text = value.strip()

        try:
            parsed = json.loads(text)
            return np.asarray(parsed, dtype=float).ravel()
        except Exception:
            parsed = ast.literal_eval(text)
            return np.asarray(parsed, dtype=float).ravel()

    raise TypeError(
        f"Tipo não reconhecido na coluna de theta: {type(value)}"
    )


def wrap_angle(values):
    """Coloca os ângulos no intervalo [-pi, pi)."""
    values = np.asarray(values, dtype=float)
    return (values + np.pi) % (2 * np.pi) - np.pi


if not DATA_PATH.is_file():
    alternatives = [
        Path("notebooks/results/merged.pkl"),
        Path.cwd() / "results" / "merged.pkl",
        Path.cwd().parent / "results" / "merged.pkl",
        Path.cwd().parent / "notebooks" / "results" / "merged.pkl",
    ]

    found = next(
        (path for path in alternatives if path.is_file()),
        None,
    )

    if found is None:
        raise FileNotFoundError(
            "Não encontrei merged.pkl. Coloque este notebook na pasta "
            "'notebooks' ou ajuste DATA_PATH."
        )

    DATA_PATH = found

df = pd.read_pickle(DATA_PATH).copy()

required_columns = [
    THETA_COLUMN,
    ENERGY_COLUMN,
    BITSTRING_COLUMN,
]

missing = [
    column for column in required_columns
    if column not in df.columns
]

if missing:
    raise KeyError(
        "Colunas ausentes no banco: " + ", ".join(missing)
    )

theta_list = df[THETA_COLUMN].map(parse_theta)

lengths = theta_list.map(len)
if lengths.nunique() != 1:
    raise ValueError(
        "Os vetores theta não possuem todos o mesmo comprimento."
    )

THETA = np.vstack(theta_list.to_numpy())
THETA_WRAPPED = wrap_angle(THETA)

ENERGY = pd.to_numeric(
    df[ENERGY_COLUMN],
    errors="coerce",
).to_numpy(dtype=float)

BITSTRING = (
    df[BITSTRING_COLUMN]
    .astype(str)
    .str.replace(" ", "", regex=False)
    .to_numpy()
)

valid = (
    np.isfinite(ENERGY)
    & np.all(np.isfinite(THETA_WRAPPED), axis=1)
)

THETA_WRAPPED = THETA_WRAPPED[valid]
ENERGY = ENERGY[valid]
BITSTRING = BITSTRING[valid]
df_valid = df.loc[valid].reset_index(drop=True)

ENERGY_GAP = np.abs(ENERGY - EXACT_ENERGY)
IS_EXACT = BITSTRING == EXACT_BITSTRING
IS_NEAR = ENERGY_GAP < NEAR_OPTIMAL_TOL

N_RUNS, N_THETAS = THETA_WRAPPED.shape

print("Arquivo carregado:", DATA_PATH.resolve())
print("Execuções válidas:", N_RUNS)
print("Parâmetros por execução:", N_THETAS)
print("Taxa do bitstring exato:", f"{100 * IS_EXACT.mean():.2f}%")
print("Taxa com gap < 1e-3:", f"{100 * IS_NEAR.mean():.2f}%")



## 3. Dividir cada parâmetro em regiões angulares

O intervalo \([-\pi,\pi)\) será dividido em 24 faixas.

Para cada \(\theta_j\) e cada faixa, calculamos:

- número de execuções;
- energia média;
- energia mediana;
- gap energético mediano;
- taxa do bitstring exato;
- taxa de soluções quase ótimas.

Faixas com menos de 30 execuções são ignoradas no ranking para evitar conclusões baseadas em poucos pontos.


In [ ]:

# ============================================================
# 3. RESUMO POR THETA E FAIXA ANGULAR
# ============================================================

bin_edges = np.linspace(
    -np.pi,
    np.pi,
    N_ANGLE_BINS + 1,
)

bin_centers = 0.5 * (
    bin_edges[:-1] + bin_edges[1:]
)

rows = []

for theta_index in range(N_THETAS):

    values = THETA_WRAPPED[:, theta_index]

    # Índice da faixa angular de cada execução.
    bin_index = np.digitize(
        values,
        bin_edges,
        right=False,
    ) - 1

    # Segurança para o limite superior.
    bin_index = np.clip(
        bin_index,
        0,
        N_ANGLE_BINS - 1,
    )

    for current_bin in range(N_ANGLE_BINS):

        mask = bin_index == current_bin
        count = int(mask.sum())

        if count == 0:
            continue

        rows.append({
            "theta_index": theta_index,
            "theta_name": f"theta_{theta_index:02d}",
            "angle_bin": current_bin,
            "angle_center_rad": float(
                bin_centers[current_bin]
            ),
            "angle_center_deg": float(
                np.degrees(bin_centers[current_bin])
            ),
            "n_runs": count,
            "mean_energy": float(
                np.mean(ENERGY[mask])
            ),
            "median_energy": float(
                np.median(ENERGY[mask])
            ),
            "median_energy_gap": float(
                np.median(ENERGY_GAP[mask])
            ),
            "mean_energy_gap": float(
                np.mean(ENERGY_GAP[mask])
            ),
            "exact_bitstring_rate": float(
                np.mean(IS_EXACT[mask])
            ),
            "near_optimal_rate": float(
                np.mean(IS_NEAR[mask])
            ),
        })

bin_summary = pd.DataFrame(rows)

bin_summary.to_csv(
    TABLE_DIR / "theta_angle_bin_summary.csv",
    index=False,
)

print("Linhas do resumo:", len(bin_summary))
bin_summary.head()



## 4. Ranking dos parâmetros

Para cada \(\theta_j\), usamos apenas faixas com amostra suficiente.

### Efeito empírico sobre a energia

\[
S_E(j)
=
\max_b\operatorname{mediana}(\Delta E\mid b)
-
\min_b\operatorname{mediana}(\Delta E\mid b).
\]

### Efeito empírico sobre o bitstring

\[
S_B(j)
=
\max_b P(x^\star\mid b)
-
\min_b P(x^\star\mid b).
\]

Valores maiores indicam que a qualidade da solução muda mais entre as regiões angulares daquele parâmetro.


In [ ]:

# ============================================================
# 4. TABELA DE SENSIBILIDADE EMPÍRICA
# ============================================================

valid_bins = bin_summary.loc[
    bin_summary["n_runs"] >= MIN_RUNS_PER_BIN
].copy()

ranking_rows = []

for theta_index, group in valid_bins.groupby(
    "theta_index",
    sort=True,
):

    best_energy_row = group.loc[
        group["median_energy_gap"].idxmin()
    ]

    worst_energy_row = group.loc[
        group["median_energy_gap"].idxmax()
    ]

    best_bitstring_row = group.loc[
        group["exact_bitstring_rate"].idxmax()
    ]

    worst_bitstring_row = group.loc[
        group["exact_bitstring_rate"].idxmin()
    ]

    energy_span = (
        worst_energy_row["median_energy_gap"]
        - best_energy_row["median_energy_gap"]
    )

    bitstring_span = (
        best_bitstring_row["exact_bitstring_rate"]
        - worst_bitstring_row["exact_bitstring_rate"]
    )

    near_span = (
        group["near_optimal_rate"].max()
        - group["near_optimal_rate"].min()
    )

    ranking_rows.append({
        "theta_index": int(theta_index),
        "theta_name": f"theta_{int(theta_index):02d}",
        "valid_bins": int(len(group)),
        "energy_gap_span": float(energy_span),
        "exact_bitstring_rate_span": float(bitstring_span),
        "near_optimal_rate_span": float(near_span),
        "best_energy_angle_rad": float(
            best_energy_row["angle_center_rad"]
        ),
        "best_energy_median_gap": float(
            best_energy_row["median_energy_gap"]
        ),
        "worst_energy_angle_rad": float(
            worst_energy_row["angle_center_rad"]
        ),
        "worst_energy_median_gap": float(
            worst_energy_row["median_energy_gap"]
        ),
        "best_bitstring_angle_rad": float(
            best_bitstring_row["angle_center_rad"]
        ),
        "best_exact_bitstring_rate": float(
            best_bitstring_row["exact_bitstring_rate"]
        ),
        "worst_bitstring_angle_rad": float(
            worst_bitstring_row["angle_center_rad"]
        ),
        "worst_exact_bitstring_rate": float(
            worst_bitstring_row["exact_bitstring_rate"]
        ),
    })

theta_ranking = pd.DataFrame(ranking_rows)

# Escalas normalizadas apenas para criar um ranking combinado legível.
energy_scale = theta_ranking[
    "energy_gap_span"
].max()

bitstring_scale = theta_ranking[
    "exact_bitstring_rate_span"
].max()

theta_ranking["energy_score"] = (
    theta_ranking["energy_gap_span"]
    / max(energy_scale, 1e-12)
)

theta_ranking["bitstring_score"] = (
    theta_ranking["exact_bitstring_rate_span"]
    / max(bitstring_scale, 1e-12)
)

theta_ranking["combined_score"] = (
    0.5 * theta_ranking["energy_score"]
    + 0.5 * theta_ranking["bitstring_score"]
)

theta_ranking = theta_ranking.sort_values(
    "combined_score",
    ascending=False,
).reset_index(drop=True)

theta_ranking.to_csv(
    TABLE_DIR / "theta_empirical_ranking.csv",
    index=False,
)

theta_ranking



# 5. Gráfico — quais parâmetros mais se associam à energia?

Uma barra maior significa que o gap energético mediano muda bastante entre as regiões angulares daquele parâmetro.


In [ ]:

# ============================================================
# 5. RANKING DA ENERGIA
# ============================================================

energy_rank = theta_ranking.sort_values(
    "energy_gap_span",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(12, 9))

bars = ax.barh(
    energy_rank["theta_name"],
    energy_rank["energy_gap_span"],
)

ax.set_xlabel(
    "Variação do gap energético mediano entre regiões angulares"
)
ax.set_ylabel("Parâmetro")
ax.set_title(
    "Quais theta mais se associam à qualidade energética?"
)
ax.grid(axis="x", alpha=0.25)

top = theta_ranking.sort_values(
    "energy_gap_span",
    ascending=False,
).head(5)

comment = ["LEITURA AUTOMÁTICA", "Maiores variações:"]

for row in top.itertuples(index=False):
    comment.append(
        f"{row.theta_name}: {row.energy_gap_span:.5f}"
    )

ax.text(
    0.98,
    0.04,
    "\n".join(comment),
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    bbox={
        "boxstyle": "round,pad=0.5",
        "facecolor": "white",
        "alpha": 0.92,
    },
)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "01_ranking_energy_gap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()



# 6. Gráfico — quais parâmetros mais se associam ao bitstring ótimo?

Uma barra maior significa que a frequência do bitstring exato muda bastante entre diferentes regiões angulares daquele parâmetro.


In [ ]:

# ============================================================
# 6. RANKING DO BITSTRING
# ============================================================

bit_rank = theta_ranking.sort_values(
    "exact_bitstring_rate_span",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(12, 9))

ax.barh(
    bit_rank["theta_name"],
    100.0 * bit_rank["exact_bitstring_rate_span"],
)

ax.set_xlabel(
    "Variação da taxa do bitstring exato entre regiões angulares (%)"
)
ax.set_ylabel("Parâmetro")
ax.set_title(
    "Quais theta mais se associam ao bitstring ótimo?"
)
ax.grid(axis="x", alpha=0.25)

top = theta_ranking.sort_values(
    "exact_bitstring_rate_span",
    ascending=False,
).head(5)

comment = ["LEITURA AUTOMÁTICA", "Maiores variações:"]

for row in top.itertuples(index=False):
    comment.append(
        f"{row.theta_name}: "
        f"{100.0 * row.exact_bitstring_rate_span:.1f} p.p."
    )

ax.text(
    0.98,
    0.04,
    "\n".join(comment),
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    bbox={
        "boxstyle": "round,pad=0.5",
        "facecolor": "white",
        "alpha": 0.92,
    },
)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "02_ranking_exact_bitstring.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()



# 7. Mapa completo — energia por região angular

Cada linha representa um \(\theta_j\).

Cada coluna representa uma região entre \(-\pi\) e \(\pi\).

O mapa mostra o gap energético mediano. Regiões vazias ou com menos de 30 execuções ficam sem valor.


In [ ]:

# ============================================================
# 7. MAPA DO GAP ENERGÉTICO
# ============================================================

energy_map = valid_bins.pivot(
    index="theta_index",
    columns="angle_bin",
    values="median_energy_gap",
).reindex(
    index=range(N_THETAS),
    columns=range(N_ANGLE_BINS),
)

fig, ax = plt.subplots(figsize=(15, 9))

image = ax.imshow(
    energy_map.to_numpy(),
    aspect="auto",
    origin="lower",
    extent=[
        -np.pi,
        np.pi,
        -0.5,
        N_THETAS - 0.5,
    ],
)

ax.set_xlabel("Região angular")
ax.set_ylabel("Índice do theta")
ax.set_title(
    "Gap energético mediano em cada região angular"
)

ax.set_xticks(
    [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
    [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
)

ax.set_yticks(
    np.arange(N_THETAS),
    [str(index) for index in range(N_THETAS)],
)

colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label("Gap energético mediano")

best_theta = theta_ranking.sort_values(
    "energy_gap_span",
    ascending=False,
).iloc[0]

ax.text(
    1.01,
    0.98,
    (
        "LEITURA AUTOMÁTICA\n"
        f"Maior contraste: {best_theta.theta_name}\n"
        f"Melhor região: "
        f"{best_theta.best_energy_angle_rad:.2f} rad\n"
        f"Pior região: "
        f"{best_theta.worst_energy_angle_rad:.2f} rad"
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    bbox={
        "boxstyle": "round,pad=0.5",
        "facecolor": "white",
        "alpha": 0.92,
    },
)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "03_heatmap_energy_gap.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()



# 8. Mapa completo — taxa do bitstring ótimo

Este mapa mostra:

\[
P(x^\star\mid\theta_j\text{ em determinada região}).
\]

Regiões com valores maiores concentram mais execuções cujo bitstring dominante foi `1001011000`.


In [ ]:

# ============================================================
# 8. MAPA DA TAXA DO BITSTRING EXATO
# ============================================================

bitstring_map = valid_bins.pivot(
    index="theta_index",
    columns="angle_bin",
    values="exact_bitstring_rate",
).reindex(
    index=range(N_THETAS),
    columns=range(N_ANGLE_BINS),
)

fig, ax = plt.subplots(figsize=(15, 9))

image = ax.imshow(
    100.0 * bitstring_map.to_numpy(),
    aspect="auto",
    origin="lower",
    vmin=0.0,
    vmax=100.0,
    extent=[
        -np.pi,
        np.pi,
        -0.5,
        N_THETAS - 0.5,
    ],
)

ax.set_xlabel("Região angular")
ax.set_ylabel("Índice do theta")
ax.set_title(
    "Frequência do bitstring ótimo em cada região angular"
)

ax.set_xticks(
    [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
    [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
)

ax.set_yticks(
    np.arange(N_THETAS),
    [str(index) for index in range(N_THETAS)],
)

colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label("Bitstring exato (%)")

best_theta = theta_ranking.sort_values(
    "exact_bitstring_rate_span",
    ascending=False,
).iloc[0]

ax.text(
    1.01,
    0.98,
    (
        "LEITURA AUTOMÁTICA\n"
        f"Maior contraste: {best_theta.theta_name}\n"
        f"Melhor região: "
        f"{best_theta.best_bitstring_angle_rad:.2f} rad\n"
        f"Taxa máxima: "
        f"{100.0 * best_theta.best_exact_bitstring_rate:.1f}%"
    ),
    transform=ax.transAxes,
    ha="left",
    va="top",
    bbox={
        "boxstyle": "round,pad=0.5",
        "facecolor": "white",
        "alpha": 0.92,
    },
)

fig.tight_layout()
fig.savefig(
    FIGURE_DIR / "04_heatmap_exact_bitstring.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()



# 9. Perfis individuais dos parâmetros mais informativos

Para os seis parâmetros com maior escore combinado, cada figura mostra:

- acima: gap energético mediano;
- abaixo: frequência do bitstring ótimo;
- tamanho dos pontos: quantidade de execuções naquela faixa.

Assim é possível verificar se uma mesma região angular é simultaneamente favorável à energia e ao bitstring.


In [ ]:

# ============================================================
# 9. PERFIS DOS THETA MAIS INFORMATIVOS
# ============================================================

top_indices = (
    theta_ranking.head(TOP_THETAS)["theta_index"]
    .astype(int)
    .tolist()
)

for theta_index in top_indices:

    group = valid_bins.loc[
        valid_bins["theta_index"] == theta_index
    ].sort_values("angle_center_rad")

    if group.empty:
        continue

    rank_row = theta_ranking.loc[
        theta_ranking["theta_index"] == theta_index
    ].iloc[0]

    point_sizes = (
        25.0
        + 130.0
        * group["n_runs"]
        / group["n_runs"].max()
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(12, 8),
        sharex=True,
    )

    axes[0].plot(
        group["angle_center_rad"],
        group["median_energy_gap"],
        marker="o",
        linewidth=2,
    )

    axes[0].scatter(
        group["angle_center_rad"],
        group["median_energy_gap"],
        s=point_sizes,
        alpha=0.35,
    )

    axes[0].axvline(
        rank_row["best_energy_angle_rad"],
        linestyle="--",
        label="Melhor região energética",
    )

    axes[0].set_ylabel("Gap energético mediano")
    axes[0].set_title(
        f"{rank_row['theta_name']}: energia e bitstring por região angular"
    )
    axes[0].grid(alpha=0.25)
    axes[0].legend()

    axes[1].plot(
        group["angle_center_rad"],
        100.0 * group["exact_bitstring_rate"],
        marker="o",
        linewidth=2,
    )

    axes[1].scatter(
        group["angle_center_rad"],
        100.0 * group["exact_bitstring_rate"],
        s=point_sizes,
        alpha=0.35,
    )

    axes[1].axvline(
        rank_row["best_bitstring_angle_rad"],
        linestyle="--",
        label="Melhor região para o bitstring",
    )

    axes[1].set_xlabel("Ângulo")
    axes[1].set_ylabel("Bitstring exato (%)")
    axes[1].set_xticks(
        [-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi],
        [r"$-\pi$", r"$-\pi/2$", "0", r"$\pi/2$", r"$\pi$"],
    )
    axes[1].grid(alpha=0.25)
    axes[1].legend()

    fig.text(
        0.98,
        0.50,
        (
            "LEITURA AUTOMÁTICA\n"
            f"Contraste de energia: "
            f"{rank_row['energy_gap_span']:.5f}\n"
            f"Contraste do bitstring: "
            f"{100.0 * rank_row['exact_bitstring_rate_span']:.1f} p.p.\n"
            f"Melhor ângulo energético: "
            f"{rank_row['best_energy_angle_rad']:.2f} rad\n"
            f"Melhor ângulo do bitstring: "
            f"{rank_row['best_bitstring_angle_rad']:.2f} rad"
        ),
        ha="right",
        va="center",
        bbox={
            "boxstyle": "round,pad=0.5",
            "facecolor": "white",
            "alpha": 0.92,
        },
    )

    fig.tight_layout(rect=[0.0, 0.0, 0.82, 1.0])

    fig.savefig(
        FIGURE_DIR / f"05_profile_theta_{theta_index:02d}.png",
        dpi=300,
        bbox_inches="tight",
    )

    plt.show()



# 10. Conclusão automática

O relatório abaixo separa:

- parâmetros mais associados à energia;
- parâmetros mais associados ao bitstring;
- parâmetros fortes nas duas métricas.

A conclusão usa apenas o banco existente. Ela deve ser escrita como **associação empírica**, e não como prova de causalidade.


In [ ]:

# ============================================================
# 10. RELATÓRIO FINAL
# ============================================================

top_energy = (
    theta_ranking.sort_values(
        "energy_gap_span",
        ascending=False,
    )
    .head(5)["theta_name"]
    .tolist()
)

top_bitstring = (
    theta_ranking.sort_values(
        "exact_bitstring_rate_span",
        ascending=False,
    )
    .head(5)["theta_name"]
    .tolist()
)

top_combined = (
    theta_ranking.head(5)["theta_name"]
    .tolist()
)

print("=" * 76)
print("CONCLUSÃO — 10.000 VETORES DO MESMO HAMILTONIANO")
print("=" * 76)
print()
print("Parâmetros mais associados à variação da energia:")
print(", ".join(top_energy))
print()
print("Parâmetros mais associados à ocorrência do bitstring ótimo:")
print(", ".join(top_bitstring))
print()
print("Parâmetros com maior evidência conjunta:")
print(", ".join(top_combined))
print()
print(
    "Interpretação correta: esses parâmetros apresentam regiões angulares "
    "com diferenças sistemáticas de energia e/ou frequência do bitstring "
    "ótimo dentro das 10.000 execuções. Como os outros parâmetros também "
    "variaram entre as execuções, o resultado identifica associação e "
    "candidatos para ablação, mas não prova efeito causal isolado."
)
print()
print("Tabelas salvas em:", TABLE_DIR.resolve())
print("Figuras salvas em:", FIGURE_DIR.resolve())

display(
    theta_ranking[
        [
            "theta_name",
            "energy_gap_span",
            "exact_bitstring_rate_span",
            "near_optimal_rate_span",
            "best_energy_angle_rad",
            "best_bitstring_angle_rad",
            "combined_score",
        ]
    ].head(15)
)
